In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from tqdm.notebook import tqdm, trange
import QuantLib as ql
from pickle import load

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

## VAE Generative Model for Yield Curves

The yield curves are generated by a VAE model that is trained in chapter 20 and is discused in depth there. The decoder is used to generate the curves from a latent space.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, input_dim, z_dim, layer_dims, use_batch_norm, use_dropout):
        super(Decoder, self).__init__()

        self.use_batch_norm = use_batch_norm
        self.use_dropout = use_dropout

        self.hidden_layers = nn.ModuleList()
        prev_dim = z_dim
        for ilayer in layer_dims:
            self.hidden_layers.append(nn.Linear(prev_dim, ilayer))
            if use_batch_norm:
                self.hidden_layers.append(nn.BatchNorm1d(ilayer))
            self.hidden_layers.append(nn.LeakyReLU())
            if use_dropout:
                self.hidden_layers.append(nn.Dropout(p = 0.25))
            prev_dim = ilayer

        self.out = nn.Linear(layer_dims[-1], input_dim)

    def forward(self, x):
        for layer in self.hidden_layers:
            x = layer(x)
        x = self.out(x)
        return x

In [ ]:
input_dim = 60
latent_dims = 2
decoder_layer_dims = [32, 64]
use_batch_norm = True
use_dropout = True
decoder = Decoder(input_dim, latent_dims, decoder_layer_dims, use_batch_norm, use_dropout)

In [ ]:
decoder_file = 'vae_decoder.pt'
saved_model = torch.load(decoder_file)

In [ ]:
decoder.load_state_dict(saved_model)

In [ ]:
decoder = decoder.to(device)

In [ ]:
scaler = load(open('scaler.pkl', 'rb'))

In [ ]:
curve_matmonths = [i for i in range(1, 61)]

Once generated the yield curve data is used to construct QuantLib zero curves. For this some data setup is requieed and a series of helper functions are defined.

In [ ]:
today = ql.Date(26, 11, 2023)
ql.Settings.instance().evaluationDate = today

In [ ]:
calendar = ql.UnitedKingdom()
bussiness_convention = ql.Following
day_count = ql.Actual365Fixed()

In [ ]:
settlement_days_SONIA = 0
SONIA = ql.OvernightIndex("SONIA", settlement_days_SONIA, ql.GBPCurrency(), calendar, day_count)

In [ ]:
def buildSONIAhelpers(tenors, ois_rates):
    OIS_helpers = []
    for i, itnr in enumerate(tenors):
        if itnr < 13:
            coupon_frequency = ql.Once
            tenor = ql.Period(itnr, ql.Months)
            rate = float(ois_rates[i])
            sq = ql.SimpleQuote(rate / 100.0)
            OIS_helpers.append(ql.OISRateHelper(settlement_days_SONIA, 
                                                tenor, ql.QuoteHandle(sq), SONIA))

        else:
            coupon_frequency = ql.Annual
            tenor = ql.Period(itnr, ql.Months)
            rate = float(ois_rates[i])
            sq = ql.SimpleQuote(rate / 100.0)
            OIS_helpers.append(ql.OISRateHelper(settlement_days_SONIA, 
                                                tenor, ql.QuoteHandle(sq), SONIA))
    return OIS_helpers

In [ ]:
def buildSONIACurve(tenors, ois_rates):
    sonia_ex = buildSONIAhelpers(curve_matmonths, ois_rates)
    yieldcurve = ql.PiecewiseLogCubicDiscount(today, sonia_ex, day_count)
    yieldcurve.enableExtrapolation()
    return yieldcurve

To generate the expected exposure profiles for 5 year interest rate swaps, the analytic solution is used. This uses the price of a strip of European swaptions. Hence a series of helper functions were defined to construct swaps, swaptions and to value a European swaption.

In [ ]:
def buildSwap(start_date, start_tnr, swap_tenor, calendar, 
              fixed_rate, curve_handle, receiver=True):
    short_conv = ql.Following
    long_conv = ql.ModifiedFollowing
    end_of_month = False
    fixed_dc = ql.Actual365Fixed()
    float_dc = ql.Actual365Fixed()
    notional = 1.0
    pay_fixed = True
    swap_start_date = start_date + start_tnr
    swap_end_date = swap_start_date + swap_tenor
    fixedLegDayCounter = ql.SimpleDayCounter()
    floatingLegDayCounter = ql.SimpleDayCounter()
    index = ql.OvernightIndex("SONIA", 0, ql.GBPCurrency(),
                       calendar, float_dc, curve_handle)
    fixedLegTenor = ql.Period(12, ql.Months)
    fixedLegAdjustment = ql.Unadjusted
    fixedLegSchedule = ql.Schedule(swap_start_date, swap_end_date, 
                                 fixedLegTenor, calendar,
                                 fixedLegAdjustment, fixedLegAdjustment,
                                 ql.DateGeneration.Forward, end_of_month)
    floatingLegTenor = ql.Period(12, ql.Months)
    floatingLegAdjustment = ql.Unadjusted
    floatingLegSchedule = ql.Schedule(swap_start_date, swap_end_date,
                                      floatingLegTenor, calendar,
                                      floatingLegAdjustment, floatingLegAdjustment,
                                      ql.DateGeneration.Forward, end_of_month)

    if receiver:
        swap_pr = ql.VanillaSwap.Receiver
    else:
        swap_pr = ql.VanillaSwap.Payer
    irs = ql.VanillaSwap(swap_pr, notional, fixedLegSchedule, 
                         fixed_rate, fixedLegDayCounter, floatingLegSchedule, 
                         index, 0, floatingLegDayCounter)
    return irs

In [ ]:
def buildSwaption(start_date, start_tnr, swap_tenor,
                  calendar, fixed_rate, curve_handle, receiver=True):
    swap = buildSwap(start_date, start_tnr, swap_tenor, 
                     calendar, fixed_rate, curve_handle, receiver)
    exercise_date = start_date + start_tnr
    exercise = ql.EuropeanExercise(exercise_date)
    settlementType = ql.Settlement.Physical 
    settlementMethod = ql.Settlement.ParYieldCurve 
    return ql.Swaption(swap, exercise, settlementType)

In [ ]:
def generateEPE(start_date, len_tnr, period, calendar, fixed_rate,
                curve_handle, vol, receiver):
    #t=0
    swap_ex = buildSwap(today, ql.Period(2, ql.Days), ql.Period(len_tnr, ql.Months), calendar, fixed_rate, zero_curve)
    swap_curve = ql.DiscountingSwapEngine(curve_handle)
    swap_ex.setPricingEngine(swap_curve)
    swap_npv = swap_ex.NPV()
    epe_lst = list()
    epe_lst.append(max(swap_npv, 0.0))
    
    cur_tnr = period
    vol_handle = ql.QuoteHandle(ql.SimpleQuote(vol))
    engine = ql.BachelierSwaptionEngine(zero_curve, vol_handle)
    while cur_tnr < len_tnr:
        swaption_ex = buildSwaption(start_date, 
                               ql.Period(cur_tnr, ql.Months), 
                               ql.Period(len_tnr - cur_tnr, ql.Months), 
                               calendar, fixed_rate, zero_curve, receiver)
        swaption_ex.setPricingEngine(engine)
        ee = swaption_ex.NPV()
        epe_lst.append(ee)
        cur_tnr = cur_tnr + period
    epe_lst.append(0.0)
    return epe_lst

## Generate 10000 yield curves

In [ ]:
samples = 10000
sample_z = torch.randn((samples, 2), device=device)
sample_curves = decoder(sample_z)
sample_curves = scaler.inverse_transform(sample_curves.detach().cpu().numpy())

For each curve, 100 random fixed rates and normal volatility values are generated. These are then used to generate expected exposure profiles.

In [ ]:
vol_lst = list()
curve_dat_lst = list()
fixed_lst = list()
epe_lst = list()
no_per_curve = 100
for icurve in sample_curves:
    crv = buildSONIACurve(curve_matmonths, icurve)
    zero_curve = ql.YieldTermStructureHandle(crv)
    for isample in range(no_per_curve):
        curve_dat_lst.append(icurve)
        # random vol
        vol = np.random.uniform(0.0, 0.025)
        vol_lst.append(vol)
        # random fixed rate
        fixed_rate = np.random.uniform(0.0, 0.1)
        fixed_lst.append(fixed_rate)
        # generate epe
        epe = generateEPE(today, 60, 6, calendar, fixed_rate, zero_curve, vol, True)
        epe_lst.append(epe)

In [ ]:
def construct_dataframe(vol_lst, curve_lst, fixed_lst, epe_lst, columns_lst):
    # Creating a list of rows
    rows = []
    for i in range(len(vol_lst)):
        row = [vol_lst[i]] + curve_lst[i].tolist() + [fixed_lst[i]] + epe_lst[i]
        rows.append(row)

    # Creating DataFrame
    df = pd.DataFrame(rows, columns=columns_lst)
    return df

In [ ]:
columns_lst = ['vol'] + curve_matmonths + ['fixed'] + [0, 6, 12, 18, 24, 30, 36, 42, 48, 54, 60]

In [ ]:
dfout = construct_dataframe(vol_lst, curve_dat_lst, fixed_lst, epe_lst, columns_lst)

In [ ]:
dfout

In [ ]:
epe_filename = 'epedataset.csv'

In [ ]:
dfout.to_csv(epe_filename)